<a href="https://colab.research.google.com/github/ElenaLuchnikova/Analyzes/blob/main/%D0%9A%D0%BE%D0%B3%D0%BE%D1%80%D1%82%D0%BD%D1%8B%D0%B9_%D0%B0%D0%BD%D0%B0%D0%BB%D0%B8%D0%B7_Retention_SQLight3_%D0%B8%D0%BD%D1%82%D0%B5%D1%80%D0%BD%D0%B5%D1%82_%D0%BC%D0%B0%D0%B3%D0%B0%D0%B7%D0%B8%D0%BD_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Когортный анализ** — это метод анализа данных, который позволяет исследовать поведение и характеристики групп людей (когорт), объединённых по определённому признаку, например, по времени начала использования продукта, даты регистрации, даты первой покупки и т.д.

Основная идея — разделить пользователей или объекты на когорты и отслеживать их поведение со временем. Это помогает понять, как изменяются показатели (например, удержание, активность, доходы) у разных групп и выявить тренды или закономерности.

Выполним с помощью SQL когортный анализ Retantion для интернет-магазина косметики и бытовой химии, возьмем продажи за 2024 г.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from google.colab import drive
import seaborn as sns
import matplotlib.pyplot as plt



In [ ]:
df = pd.read_csv(
    '/content/drive/MyDrive/data_store.csv',
    encoding='1251',
    sep=';')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52286 entries, 0 to 52285
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   user    52286 non-null  int64 
 1   dt      52286 non-null  object
 2   sale    52286 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.2+ MB


In [ ]:
df.head()

,user,dt,sale
0,1,27.02.2024,2763
1,1,28.02.2024,5872
2,1,01.03.2024,540
3,1,02.03.2024,3535
4,1,04.03.2024,1256


In [ ]:

df['dt']=pd.to_datetime(df['dt'], format="%d.%m.%Y")

In [ ]:
# Создаём подключение к временной базе данных в памяти
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

In [ ]:
# Создаем таблицу с учетом типов данных
cursor.execute('''
CREATE TABLE data_store  (

    user INT,

    dt DATE,

    sale REAL

)
''')

In [ ]:
# Вставляем данные из DataFrame в таблицу
df.to_sql('data_store', conn, if_exists='append', index=False)

52286

In [ ]:
query = "select *  from data_store limit 5"
result = pd.read_sql_query(query, conn)
print(result)

   user                   dt    sale
0     1  2024-02-27 00:00:00  2763.0
1     1  2024-02-28 00:00:00  5872.0
2     1  2024-03-01 00:00:00   540.0
3     1  2024-03-02 00:00:00  3535.0
4     1  2024-03-04 00:00:00  1256.0


# Когортный анализ retantion

**Выполним когортный анализ Retention:**


1)для каждого клиента определим его когорту (в данном случае - месяц и год самой первой покупки)

2)сформируем таблицу, в которой выделим периоды: 0, 1, 3, 7, 14, 30,60,90,180, 270 и 365 дней с момента первой покупки и посчитаем, сколько клиентов вернулось в каждый из дней


In [ ]:
query='''
with t1 as (
	select
		dt,
		user,
		sale,
		-- Формируем когорту, находя самую первую покупку для каждого клиента
		-- и извлекая из нее месяц и год
		strftime('%Y-%m',
            first_value(dt) over(partition by user order by dt)
        ) as cohort,
        first_value(dt) over(partition by user order by dt) as first_day,
        julianday(dt) - julianday(first_value(dt) over(partition by user order by dt)) as diff

	from data_store
	where dt between '2024-01-01' and '2025-01-01'),

  t2 as (select cohort,
  diff,
  count(distinct user) as cnt
  from t1
  where diff in (0,1,3,7, 14,30,60,90,180,270,365)
  group by cohort, diff
  order by cohort, diff)



  select cohort,
  max(case when diff=0 then cnt end) as "0",
  max(case when diff=1 then cnt end) as "1",
  max(case when diff=3 then cnt end) as "3",
  max(case when diff=7 then cnt end) as "7",
  max(case when diff=14 then cnt end) as "14",
  max(case when diff=30 then cnt end) as "30",
  max(case when diff=60 then cnt end) as "60",
  max(case when diff=90 then cnt end) as "90",
  max(case when diff=180 then cnt end) as "180",
  max(case when diff=270 then cnt end) as "270",
  max(case when diff=360 then cnt end) as "360"
  from t2
  group by cohort
  '''

# Получаем результат
result = pd.read_sql_query(query, conn)
# Выводим результат
print(result)

    cohort    0      1      3      7    14    30    60    90   180  270   360
0  2024-01  811  149.0  130.0  130.0  81.0  85.0  40.0  35.0  12.0  NaN  None
1  2024-02  208   66.0   73.0   56.0  37.0  43.0  15.0  19.0   3.0  1.0  None
2  2024-03  170   67.0   74.0   65.0  27.0  20.0  24.0  14.0   1.0  NaN  None
3  2024-04  124   51.0   55.0   48.0  24.0  27.0  10.0  11.0   2.0  NaN  None
4  2024-05  152   78.0   73.0   64.0  40.0  30.0  17.0  14.0   4.0  NaN  None
5  2024-06  108   59.0   58.0   60.0  28.0  18.0  22.0  10.0   5.0  NaN  None
6  2024-07    3    NaN    NaN    NaN   NaN   NaN   NaN   NaN   NaN  NaN  None
7  2024-09    1    NaN    NaN    NaN   NaN   NaN   NaN   NaN   NaN  NaN  None


In [ ]:
res=pd.DataFrame(result)
res.set_index(res.columns[0], inplace=True)

In [ ]:
res

,0,1,3,7,14,30,60,90,180,270,300
cohort,,,,,,,,,,,
2024-01,811,149.0,130.0,130.0,81.0,85.0,40.0,35.0,12.0,NaN,None
2024-02,208,66.0,73.0,56.0,37.0,43.0,15.0,19.0,3.0,1.0,None
2024-03,170,67.0,74.0,65.0,27.0,20.0,24.0,14.0,1.0,NaN,None
2024-04,124,51.0,55.0,48.0,24.0,27.0,10.0,11.0,2.0,NaN,None
2024-05,152,78.0,73.0,64.0,40.0,30.0,17.0,14.0,4.0,NaN,None
2024-06,108,59.0,58.0,60.0,28.0,18.0,22.0,10.0,5.0,NaN,None
2024-07,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
2024-09,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None


In [ ]:
styled_res = res.style.background_gradient(cmap='YlGnBu',vmin=0, vmax=812)
styled_res

,0,1,3,7,14,30,60,90,180,270,300
cohort,,,,,,,,,,,
2024-01,811,149.000000,130.000000,130.000000,81.000000,85.000000,40.000000,35.000000,12.000000,nan,None
2024-02,208,66.000000,73.000000,56.000000,37.000000,43.000000,15.000000,19.000000,3.000000,1.000000,None
2024-03,170,67.000000,74.000000,65.000000,27.000000,20.000000,24.000000,14.000000,1.000000,nan,None
2024-04,124,51.000000,55.000000,48.000000,24.000000,27.000000,10.000000,11.000000,2.000000,nan,None
2024-05,152,78.000000,73.000000,64.000000,40.000000,30.000000,17.000000,14.000000,4.000000,nan,None
2024-06,108,59.000000,58.000000,60.000000,28.000000,18.000000,22.000000,10.000000,5.000000,nan,None
2024-07,3,nan,nan,nan,nan,nan,nan,nan,nan,nan,None
2024-09,1,nan,nan,nan,nan,nan,nan,nan,nan,nan,None


* Когорты Июля и Сентября включают в себя 1-3 покупателя (это разовые клиенты), поэтому мы их анализировать не будем.
*  Видим сильный спад по всем когортам уже в первые дни. Но в контексте сферы продаж магазина (магазин продает косметику и бытовую химию - товары не каждодневного спроса), это не  критично.
Гораздо более тревожный факт - постепенное снижение возвратов покупателей в долгосрочной перспективе - на 180й день возвращается только 1-12 покупателей. К 270 дню почти все покупатели отваливаются.

*  Самая большая когорта - Январская когорта. Скорее всего в этом месяце проходила маркетинговая компания (например, новогодние распродажи, акции и пр.). Но клиенты постепенно начали отваливаться, мало повторных покупок. Большинству клиентов этой когорты не интересен магазин и продукция, много разовых покупателей, привлеченных большой скидкой (возможно, привлекается дешевый и некачественный трафик аудитории)


*   В остальных когортах количество привлеченных покупателей заметно меньше.
Вероятно, были изменения в маркетинговой и рекламной кампаниях (стали предлагать меньше скидок, акций, меньше давать рекламы или рекламная аудитория оказалась менее релевантной). И также покупатели постепенно отваливаются.

*  Наиболее стабильными когортами оказались Апрельская, Майская и  Июньская. В самом начале привлечено было не так много клиентов (124, 152 и 108). Но соотношение сохранившихся покупателей к изначальному числу (по месяцам) лучше, чем в других когортах. Возможно, лучше была настроена рекламная компания, которая привлекла больше именно постоянных клиентов, которым нравится сам магазин и продукция. Следует проанализировать настройки майских и июньских рекламных компаний, понять, почему они были успешны в плане формирования ядра постоянных покупателей. Посмотреть, какие рекламные каналы использовались.


**В плане привлечения клиентов лучше себя показала когорта Января. А если говорить об удержании - то когорты Апреля, Мая и Июня. В целом же можно сказать, что у магазина есть общая проблема с удержанием клиентов, проблемы с повторными покупками. Нужно поработать с качеством привлекаемых клиентов, с лояльностью покупателей. Я бы порекомендовала ввести программы лояльности (бонусную программу, кэшбэк за покупки), скидки на повторные покупки (можно с ограничением по времени), рассылки, персональные подборки товаров, бесплатную доставку на повторную покупку, если это еще не реализовано.
Также нужно проанализировать технические моменты, может быть у магазина неудобный сайт, сложности с оплатой, неудобный поиск товаров. Еще можно проанализировать цены у магазинов-конкурентов (возможно новые клиенты покупают первый раз только благодаря большой скидке за первый заказ, а совершать повторные покупки для них дорого и они уходят к конкурентам).**







